# 연금저축 연 납입액 계산기

현재 나이와 연금저축 잔액, 연금 개시 나이, 목표 금액, 예상 수익률을 입력한 뒤 아래 코드 셀을 실행하세요. 매월 말 필요한 납입액과 그 연간 합계를 계산합니다.

In [ ]:
# @title 연금저축 목표를 입력하고 실행하세요
현재나이 = 30  # @param {type:"integer", min:1, max:79, step:1}
현재연금저축잔액_만원 = 0  # @param {type:"number", min:0, step:100}
연금개시나이 = 55  # @param {type:"integer", min:2, max:80, step:1}
목표연금저축액_억원 = 2.0  # @param {type:"number", min:0.01, step:0.1}
예상연수익률_퍼센트 = 10.0  # @param {type:"number", min:0, step:0.1}

"""목표 연금저축액에 필요한 월말 납입액을 월복리로 역산한다.

현재 잔액은 계산 시작 시점에 투자하며, 목표액은 연금 개시 직전에 평가한다.
세금, 수수료, 상품 보수, 물가와 수익률 변동은 반영하지 않는다.
"""

import math

import pandas as pd
from IPython.display import HTML, display

WON_PER_EOK = 100_000_000
WON_PER_MANWON = 10_000
MONTHS_PER_YEAR = 12
CONTRIBUTION_ROUNDING_KRW = 1_000


def validate_parameters(
    current_age: int,
    pension_age: int,
    current_balance_krw: float,
    target_asset_krw: float,
    annual_return_rate: float,
) -> None:
    """계산에 사용하는 입력값의 범위를 검증한다."""
    if current_age < 1:
        raise ValueError("현재 나이는 1세 이상이어야 합니다.")
    if pension_age <= current_age:
        raise ValueError("연금 개시 나이는 현재 나이보다 많아야 합니다.")
    if current_balance_krw < 0:
        raise ValueError("현재 연금저축 잔액은 0원 이상이어야 합니다.")
    if target_asset_krw <= 0:
        raise ValueError("목표 연금저축액은 0원보다 커야 합니다.")
    if annual_return_rate < 0:
        raise ValueError("예상 연 수익률은 0% 이상이어야 합니다.")


def calculate_month_end_contribution(
    remaining_target_krw: float,
    periodic_return_rate: float,
    number_of_contributions: int,
) -> float:
    """기간 말 납입 조건에서 회당 필요 납입액을 계산한다."""
    if remaining_target_krw <= 0:
        return 0.0
    if periodic_return_rate == 0:
        return remaining_target_krw / number_of_contributions
    ordinary_annuity_factor = (
        ((1 + periodic_return_rate) ** number_of_contributions - 1)
        / periodic_return_rate
    )
    return remaining_target_krw / ordinary_annuity_factor


def round_up_contribution(amount_krw: float) -> float:
    """목표액이 부족하지 않도록 납입액을 1천 원 단위로 올림한다."""
    return (
        math.ceil(amount_krw / CONTRIBUTION_ROUNDING_KRW)
        * CONTRIBUTION_ROUNDING_KRW
    )


def format_manwon(amount_krw: float) -> str:
    """원화 금액을 만 원 단위 문자열로 표시한다."""
    if amount_krw == 0:
        return "0만 원"
    return f"{amount_krw / WON_PER_MANWON:,.1f}만 원"


def calculate_monthly_schedule(
    current_age: int,
    current_balance_krw: float,
    monthly_contribution_krw: float,
    monthly_return_rate: float,
    saving_months: int,
) -> pd.DataFrame:
    """월별 누적 원금, 누적 수익과 총 잔액을 계산한다."""
    balance_krw = current_balance_krw
    cumulative_principal_krw = current_balance_krw
    rows = []
    for month_number in range(1, saving_months + 1):
        beginning_balance_krw = balance_krw
        interest_krw = beginning_balance_krw * monthly_return_rate
        balance_krw = (
            beginning_balance_krw + interest_krw + monthly_contribution_krw
        )
        cumulative_principal_krw += monthly_contribution_krw
        cumulative_return_krw = balance_krw - cumulative_principal_krw
        attained_age_months = current_age * MONTHS_PER_YEAR + month_number
        attained_age, age_month = divmod(
            attained_age_months, MONTHS_PER_YEAR
        )
        rows.append({
            "월차": month_number,
            "나이": f"{attained_age}세 {age_month}개월",
            "누적 원금": cumulative_principal_krw,
            "누적 수익": cumulative_return_krw,
            "총 잔액": balance_krw,
        })
    return pd.DataFrame(rows)


def format_schedule_table(schedule: pd.DataFrame) -> str:
    """월별 적립 내역을 화면 표시용 HTML 표로 변환한다."""
    display_schedule = schedule.copy()
    money_columns = ("누적 원금", "누적 수익", "총 잔액")
    for column in money_columns:
        display_schedule[column] = display_schedule[column].map(format_manwon)
    return display_schedule.to_html(
        index=False, border=0, classes="schedule-table", escape=True
    )


def main() -> None:
    """필요한 월말 납입액과 월 납입액의 연간 합계를 표시한다."""
    current_age = int(현재나이)
    pension_age = int(연금개시나이)
    current_balance_krw = float(현재연금저축잔액_만원) * WON_PER_MANWON
    target_asset_krw = float(목표연금저축액_억원) * WON_PER_EOK
    annual_return_rate = float(예상연수익률_퍼센트) / 100
    validate_parameters(
        current_age,
        pension_age,
        current_balance_krw,
        target_asset_krw,
        annual_return_rate,
    )

    saving_years = pension_age - current_age
    saving_months = saving_years * MONTHS_PER_YEAR
    current_balance_future_value_krw = (
        current_balance_krw * (1 + annual_return_rate) ** saving_years
    )
    remaining_target_krw = max(
        target_asset_krw - current_balance_future_value_krw,
        0.0,
    )
    monthly_return_rate = (
        (1 + annual_return_rate) ** (1 / MONTHS_PER_YEAR) - 1
    )
    monthly_contribution_krw = round_up_contribution(
        calculate_month_end_contribution(
            remaining_target_krw, monthly_return_rate, saving_months
        )
    )
    annual_contribution_krw = monthly_contribution_krw * MONTHS_PER_YEAR
    monthly_schedule = calculate_monthly_schedule(
        current_age,
        current_balance_krw,
        monthly_contribution_krw,
        monthly_return_rate,
        saving_months,
    )
    schedule_table = format_schedule_table(monthly_schedule)
    ending_balance_krw = float(monthly_schedule.iloc[-1]["총 잔액"])

    display(HTML(f"""
    <style>
    .pension-card{{max-width:680px;padding:24px 26px;border:1px solid #f0f2f5;
      border-radius:14px;background:#fff;font-family:Pretendard,-apple-system,
      BlinkMacSystemFont,"Segoe UI",sans-serif;color:#1e293b}}
    .pension-brand{{margin:0 0 6px;color:#64748b;font-size:13px}}
    .pension-title{{margin:0 0 18px;color:#0b0b0b;font-size:21px;font-weight:700}}
    .pension-conditions{{display:grid;grid-template-columns:1fr 1fr;gap:0;
      margin-bottom:18px;border-top:1px solid #f0f2f5}}
    .pension-condition{{padding:9px 4px;border-bottom:1px solid #f0f2f5;
      font-size:14px}}
    .pension-condition:nth-child(even){{text-align:right}}
    .pension-results{{display:grid;grid-template-columns:1fr 1fr;gap:12px}}
    .pension-result{{padding:16px;border-radius:10px;background:#f3f7fd}}
    .pension-label{{margin:0 0 5px;color:#64748b;font-size:13px}}
    .pension-value{{margin:0;color:#2f7dd3;font-size:22px;font-weight:700;
      font-variant-numeric:tabular-nums}}
    .pension-note{{margin:14px 0 0;color:#64748b;font-size:13px;line-height:1.55}}
    .schedule-details{{max-width:680px;margin-top:14px;border:1px solid #f0f2f5;
      border-radius:12px;background:#fff;font-family:Pretendard,-apple-system,
      BlinkMacSystemFont,"Segoe UI",sans-serif;color:#1e293b}}
    .schedule-summary{{padding:15px 18px;cursor:pointer;color:#2b4a75;
      font-size:15px;font-weight:700}}
    .schedule-caption{{margin:0;padding:0 18px 12px;color:#64748b;
      font-size:13px;line-height:1.5}}
    .schedule-wrap{{max-height:520px;overflow:auto;border-top:1px solid #f0f2f5}}
    .schedule-table{{width:100%;border-collapse:separate;border-spacing:0;
      font-size:12px;font-variant-numeric:tabular-nums;white-space:nowrap}}
    .schedule-table th{{position:sticky;top:0;padding:10px 8px;background:#2b4a75;
      color:#fff;text-align:center;z-index:1}}
    .schedule-table td{{padding:9px 8px;border-bottom:1px solid #f0f2f5;
      text-align:right}}
    .schedule-table td:nth-child(1),.schedule-table td:nth-child(2){{text-align:center}}
    .schedule-table tbody tr:nth-child(even) td{{background:#fafbfc}}
    </style>
    <section class="pension-card">
      <p class="pension-brand">대도시 연구실</p>
      <h2 class="pension-title">연금저축 연 납입액 계산기</h2>
      <div class="pension-conditions">
        <div class="pension-condition">현재 나이: {current_age}세</div>
        <div class="pension-condition">현재 잔액: {현재연금저축잔액_만원:g}만 원</div>
        <div class="pension-condition">연금 개시: {pension_age}세</div>
        <div class="pension-condition">목표 금액: {목표연금저축액_억원:g}억 원</div>
        <div class="pension-condition">적립 기간: {saving_years}년</div>
        <div class="pension-condition">예상 연 수익률: {예상연수익률_퍼센트:g}%</div>
      </div>
      <div class="pension-results">
        <div class="pension-result">
          <p class="pension-label">필요 월 납입액</p>
          <p class="pension-value">{format_manwon(monthly_contribution_krw)}</p>
        </div>
        <div class="pension-result">
          <p class="pension-label">연 납입액</p>
          <p class="pension-value">{format_manwon(annual_contribution_krw)}</p>
        </div>
      </div>
      <p class="pension-note">
        ※ 현재 잔액은 계산 시작 시점에 투자하고, 이후 매월 말 추가 납입한다고 가정했습니다.<br>
        ※ 연 납입액은 월 납입액의 12개월 합계입니다.<br>
        ※ 연 수익률을 월복리로 환산했으며, 월 납입액은 1천 원 단위로 올림했습니다.<br>
        ※ 세금·수수료·상품 보수·물가와 수익률 변동은 제외했습니다.
      </p>
    </section>
    <details class="schedule-details">
      <summary class="schedule-summary">
        {current_age}세부터 {pension_age}세까지 월별 적립 내역 펼쳐보기
      </summary>
      <p class="schedule-caption">
        현재 잔액과 월말 납입액을 누적 원금으로, 총 잔액과 누적 원금의 차이를 누적 수익으로 표시합니다.<br>
        {pension_age}세 예상 잔액은 <strong>{format_manwon(ending_balance_krw)}</strong>입니다.
        월 납입액을 1천 원 단위로 올림하여 목표액을 소폭 초과할 수 있습니다.
      </p>
      <div class="schedule-wrap">{schedule_table}</div>
    </details>
    """))


main()
